# RL Project — Highway-v0
**Core Task**: Custom DQN + Stable-Baselines3 PPO on `highway-v0`  
**Extension**: Double DQN (DDQN) vs vanilla DQN ablation


In [ ]:
!pip install highway-env

In [3]:
import os
import math
import random
import time
import statistics as stats
from collections import namedtuple, deque
from dataclasses import dataclass, asdict
from itertools import count

import gymnasium as gym
import highway_env          # registers highway-v0 with gymnasium
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from shared_core_config import SHARED_CORE_CONFIG, SHARED_CORE_ENV_ID

# IPython display helpers for live plots inside Jupyter
is_ipython = "inline" in matplotlib.get_backend()
if is_ipython:
    from IPython import display
plt.ion()

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print("device:", device)

device: cpu


## Environment

In [4]:
def make_env(seed: int, render: bool = False) -> gym.Env:
    """Create and seed a highway-v0 environment.

    Args:
        seed:   RNG seed forwarded to env.reset / action_space / obs_space.
        render: When True the env renders offscreen (needed for video recording).
    """
    config = dict(SHARED_CORE_CONFIG)
    render_mode = None
    if render:
        config["offscreen_rendering"] = True
        render_mode = "rgb_array"

    env = gym.make(SHARED_CORE_ENV_ID, config=config, render_mode=render_mode)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return env

## DQN Configuration

In [5]:
@dataclass
class DQNConfig:
    # ── Replay buffer ──────────────────────────────────────────────────────
    replay_capacity: int  = 15_000  # max transitions stored
    batch_size:      int  = 32      # mini-batch size for each gradient step
    learning_starts: int  = 200     # steps before gradient updates begin

    # ── Discount & target network ──────────────────────────────────────────
    gamma:               float = 0.8   # reward discount factor
    target_update_every: int   = 50    # steps between target-net syncs (hard update)

    # ── ε-greedy exploration schedule ─────────────────────────────────────
    eps_start: float = 1.0    # initial exploration rate
    eps_end:   float = 0.05   # final (minimum) exploration rate
    eps_decay: int   = 5_000  # exponential decay half-life in steps

    # ── Optimiser ──────────────────────────────────────────────────────────
    lr:          float = 5e-4  # AdamW learning rate
    hidden_size: int   = 256   # width of each hidden layer

    # ── Training budget ────────────────────────────────────────────────────
    num_episodes:      int = int(2e4)  # total episodes per seed
    eval_episodes:     int = 25        # episodes used for periodic evaluation
    checkpoint_every:  int = 100       # save a checkpoint every N episodes
    best_avg_window:   int = 20        # window for tracking best checkpoint

## Replay Buffer & Network Architecture

In [6]:
Transition = namedtuple("Transition", ("state", "action", "next_state", "reward"))


class ReplayBuffer:
    """Fixed-size circular buffer of (s, a, s', r) transitions."""

    def __init__(self, capacity: int):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)


class Net(nn.Module):
    """Two-hidden-layer MLP mapping observations to Q-values.

    Input  : flattened kinematics observation  (vehicles_count × features)
    Output : Q(s, a) for every discrete action
    """

    def __init__(self, n_obs: int, n_actions: int, hidden_size: int = 256):
        super().__init__()
        self.layer1 = nn.Linear(n_obs, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.layer3 = nn.Linear(hidden_size, n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

## Utilities

In [7]:
def set_seed(seed: int):
    """Seed Python random, NumPy, and PyTorch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_state_tensor(obs) -> torch.Tensor:
    """Convert a numpy observation to a (1, n_obs) float tensor on device."""
    return torch.tensor(obs, dtype=torch.float32, device=device).view(1, -1)

In [8]:
RUN_ID         = time.strftime("%Y%m%d-%H%M%S")
CHECKPOINT_DIR = os.path.join("checkpoints", f"run_{RUN_ID}")
print("Checkpoint dir:", CHECKPOINT_DIR)


def save_checkpoint(path: str, agent, episode: int, episode_rewards: list):
    """Persist policy / target weights, optimiser state, and metadata."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(
        {
            "policy_state_dict":    agent.policy_net.state_dict(),
            "target_state_dict":    agent.target_net.state_dict(),
            "optimizer_state_dict": agent.optimizer.state_dict(),
            "steps_done":           agent.steps_done,
            "episode":              episode,
            "seed":                 agent.seed,
            "episode_rewards":      episode_rewards,
            "config":               asdict(agent.cfg),
        },
        path,
    )

Checkpoint dir: checkpoints\run_20260326-145418


## DQN Agent

In [9]:
class DQNAgent:
    """Vanilla DQN agent (Mnih et al., 2015).

    Uses:
    - ε-greedy exploration with exponential decay
    - Experience replay buffer
    - Hard target-network synchronisation every `target_update_every` steps
    - Huber (SmoothL1) loss with gradient clipping
    """

    def __init__(self, env: gym.Env, cfg: DQNConfig, seed: int):
        self.env  = env
        self.cfg  = cfg
        self.seed = seed

        set_seed(seed)

        state, _ = env.reset(seed=seed)
        self.n_observations = int(torch.tensor(state).numel())
        self.n_actions      = env.action_space.n

        self.policy_net = Net(self.n_observations, self.n_actions, cfg.hidden_size).to(device)
        self.target_net = Net(self.n_observations, self.n_actions, cfg.hidden_size).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()  # target net is never trained directly

        self.optimizer  = optim.AdamW(self.policy_net.parameters(), lr=cfg.lr, amsgrad=True)
        self.memory     = ReplayBuffer(cfg.replay_capacity)
        self.steps_done = 0

    # ── Action selection ────────────────────────────────────────────────────

    def select_action(self, state: torch.Tensor) -> torch.Tensor:
        """ε-greedy action: exploit policy_net or explore uniformly."""
        eps = self.cfg.eps_end + (self.cfg.eps_start - self.cfg.eps_end) * \
              math.exp(-self.steps_done / self.cfg.eps_decay)
        self.steps_done += 1

        if random.random() > eps:
            with torch.no_grad():
                return self.policy_net(state).max(1).indices.view(1, 1)
        return torch.tensor([[self.env.action_space.sample()]], device=device, dtype=torch.long)

    # ── One gradient step ───────────────────────────────────────────────────

    def optimize(self):
        """Sample a mini-batch and perform one gradient update on policy_net."""
        if len(self.memory) < max(self.cfg.batch_size, self.cfg.learning_starts):
            return

        transitions = self.memory.sample(self.cfg.batch_size)
        batch = Transition(*zip(*transitions))

        # Mask for non-terminal transitions
        non_final_mask = torch.tensor(
            [s is not None for s in batch.next_state], device=device, dtype=torch.bool
        )
        non_final_next = [s for s in batch.next_state if s is not None]
        non_final_next = torch.cat(non_final_next) if non_final_next else \
                         torch.empty((0, self.n_observations), device=device)

        state_batch  = torch.cat(batch.state)
        action_batch = torch.cat(batch.action)
        reward_batch = torch.cat(batch.reward)

        # Q(s, a) from the policy network
        q_values = self.policy_net(state_batch).gather(1, action_batch)

        # Bootstrap target: r + γ * max_a' Q_target(s', a')
        next_q = torch.zeros(self.cfg.batch_size, device=device)
        with torch.no_grad():
            if non_final_next.numel() > 0:
                next_q[non_final_mask] = self.target_net(non_final_next).max(1).values

        target = (next_q * self.cfg.gamma + reward_batch).unsqueeze(1)

        loss = F.smooth_l1_loss(q_values, target)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()

    # ── Training loop ───────────────────────────────────────────────────────

    def train(self, plot_fn=None):
        """Run full training loop.

        Returns:
            episode_rewards : list of total rewards per episode
            episode_lengths : list of step counts per episode
        """
        episode_rewards  = []
        episode_lengths  = []
        best_avg_reward  = float("-inf")

        for i_episode in range(self.cfg.num_episodes):
            state, _ = self.env.reset(seed=self.seed + i_episode)
            state     = to_state_tensor(state)
            total_r   = 0.0

            for t in count():
                action = self.select_action(state)
                obs, reward, terminated, truncated, _ = self.env.step(action.item())
                total_r += reward

                done       = terminated or truncated
                next_state = None if done else to_state_tensor(obs)

                self.memory.push(state, action, next_state, torch.tensor([reward], device=device))
                state = next_state

                self.optimize()

                if self.steps_done % self.cfg.target_update_every == 0:
                    self.target_net.load_state_dict(self.policy_net.state_dict())

                if done:
                    episode_rewards.append(total_r)
                    episode_lengths.append(t + 1)
                    if plot_fn:
                        plot_fn(episode_rewards)
                    break

            # ── Periodic checkpoint ────────────────────────────────────────
            if (i_episode + 1) % self.cfg.checkpoint_every == 0:
                path = os.path.join(CHECKPOINT_DIR, f"seed{self.seed}_ep{i_episode + 1}.pt")
                save_checkpoint(path, self, i_episode + 1, episode_rewards)

            # ── Best-model checkpoint ──────────────────────────────────────
            if len(episode_rewards) >= self.cfg.best_avg_window:
                window_avg = stats.mean(episode_rewards[-self.cfg.best_avg_window:])
                if window_avg > best_avg_reward:
                    best_avg_reward = window_avg
                    best_path = os.path.join(CHECKPOINT_DIR, f"seed{self.seed}_best.pt")
                    save_checkpoint(best_path, self, i_episode + 1, episode_rewards)

        return episode_rewards, episode_lengths

## Extension — Double DQN (DDQN)

**Hypothesis**: Vanilla DQN overestimates Q-values due to the max operator selecting and evaluating the same network.  
Double DQN (van Hasselt et al., 2016) decouples action *selection* (policy net) from action *evaluation* (target net), reducing this bias.

The only change from vanilla DQN is in the bootstrap target:
- **DQN**  : `y = r + γ · max_a' Q_target(s', a')`
- **DDQN** : `y = r + γ · Q_target(s', argmax_a' Q_policy(s', a'))`


In [10]:
class DDQNAgent(DQNAgent):
    """Double DQN agent — inherits everything from DQNAgent except the Bellman target."""

    def optimize(self):
        if len(self.memory) < max(self.cfg.batch_size, self.cfg.learning_starts):
            return

        transitions = self.memory.sample(self.cfg.batch_size)
        batch = Transition(*zip(*transitions))

        non_final_mask = torch.tensor(
            [s is not None for s in batch.next_state], device=device, dtype=torch.bool
        )
        non_final_next = [s for s in batch.next_state if s is not None]
        non_final_next = torch.cat(non_final_next) if non_final_next else \
                         torch.empty((0, self.n_observations), device=device)

        state_batch  = torch.cat(batch.state)
        action_batch = torch.cat(batch.action)
        reward_batch = torch.cat(batch.reward)

        q_values = self.policy_net(state_batch).gather(1, action_batch)

        next_q = torch.zeros(self.cfg.batch_size, device=device)
        with torch.no_grad():
            if non_final_next.numel() > 0:
                # DDQN: policy_net selects the best action, target_net evaluates it
                best_actions = self.policy_net(non_final_next).max(1).indices.unsqueeze(1)
                next_q[non_final_mask] = self.target_net(non_final_next).gather(1, best_actions).squeeze(1)

        target = (next_q * self.cfg.gamma + reward_batch).unsqueeze(1)
        loss   = F.smooth_l1_loss(q_values, target)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()

## Plotting Utilities

In [11]:
def plot_training(rewards: list, show_result: bool = False):
    """Live training curve with 100-episode rolling average."""
    plt.figure(1)
    t = torch.tensor(rewards, dtype=torch.float)
    plt.clf()
    plt.title("Result" if show_result else "Training…")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.plot(t.numpy(), alpha=0.4, label="per-episode")

    if len(t) >= 100:
        means = t.unfold(0, 100, 1).mean(1)
        pad   = torch.full((99,), float("nan"))
        plt.plot(torch.cat((pad, means)).numpy(), label="100-ep avg")
        plt.legend()

    plt.pause(0.001)
    if is_ipython:
        display.display(plt.gcf())
        if not show_result:
            display.clear_output(wait=True)


def plot_multi_seed(all_rewards: list[list], labels: list[str], title: str = "Training curves"):
    """Overlay training curves for multiple seeds, with shaded std band."""
    fig, ax = plt.subplots(figsize=(10, 4))
    max_ep = max(len(r) for r in all_rewards)
    padded = np.full((len(all_rewards), max_ep), np.nan)
    for i, r in enumerate(all_rewards):
        padded[i, :len(r)] = r

    mean = np.nanmean(padded, axis=0)
    std  = np.nanstd(padded, axis=0)
    xs   = np.arange(max_ep)

    ax.plot(xs, mean, label="mean")
    ax.fill_between(xs, mean - std, mean + std, alpha=0.2, label="±1 std")
    for i, (r, lbl) in enumerate(zip(all_rewards, labels)):
        ax.plot(r, alpha=0.3, linewidth=0.8, label=lbl)

    ax.set_title(title)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## Evaluation

In [12]:
def evaluate_policy(env: gym.Env, policy_net: nn.Module,
                    n_episodes: int, seed_offset: int) -> tuple[float, float, float]:
    """Greedy rollouts — returns (mean_reward, std_reward, mean_length).

    Uses 50 episodes by default (as required by the project spec).
    """
    policy_net.eval()
    rewards, lengths = [], []

    with torch.no_grad():
        for i in range(n_episodes):
            state, _ = env.reset(seed=seed_offset + i)
            state     = to_state_tensor(state)
            total_r   = 0.0

            for t in count():
                action = policy_net(state).max(1).indices.view(1, 1)
                obs, reward, terminated, truncated, _ = env.step(action.item())
                total_r += reward
                done     = terminated or truncated

                if done:
                    rewards.append(total_r)
                    lengths.append(t + 1)
                    break
                state = to_state_tensor(obs)

    policy_net.train()
    return stats.mean(rewards), stats.stdev(rewards), stats.mean(lengths)

## Stable-Baselines3 Baseline (PPO)

We train a PPO agent using SB3 on the same shared config as a strong baseline.


In [13]:
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_util import make_vec_env
    from stable_baselines3.common.vec_env import VecEnv
    SB3_AVAILABLE = True
except ImportError:
    SB3_AVAILABLE = False
    print("stable_baselines3 not installed — skipping SB3 cell.")
    print("Install with: pip install stable-baselines3")

if SB3_AVAILABLE:
    SB3_SEED       = 42
    SB3_TIMESTEPS  = 500_000
    SB3_EVAL_EPS   = 50
    SB3_CKPT_PATH  = os.path.join(CHECKPOINT_DIR, "ppo_highway")

    # SB3 needs a vectorised env; n_envs=1 keeps results comparable
    def _make_sb3_env():
        env = gym.make(SHARED_CORE_ENV_ID, config=dict(SHARED_CORE_CONFIG))
        env.reset(seed=SB3_SEED)
        return env

    vec_env = make_vec_env(_make_sb3_env, n_envs=4, seed=SB3_SEED)

    ppo_model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        seed=SB3_SEED,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        gamma=0.8,           # same discount as DQN for fair comparison
        tensorboard_log=os.path.join(CHECKPOINT_DIR, "tb_ppo"),
    )

    ppo_model.learn(total_timesteps=SB3_TIMESTEPS)
    ppo_model.save(SB3_CKPT_PATH)
    print("PPO checkpoint saved to:", SB3_CKPT_PATH)

    # ── Evaluate PPO ──────────────────────────────────────────────────────
    ppo_eval_env = _make_sb3_env()
    ppo_rewards  = []
    obs, _       = ppo_eval_env.reset(seed=SB3_SEED + 9999)

    for ep in range(SB3_EVAL_EPS):
        obs, _ = ppo_eval_env.reset(seed=SB3_SEED + 9999 + ep)
        done    = False
        total_r = 0.0
        while not done:
            action, _ = ppo_model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = ppo_eval_env.step(int(action))
            total_r  += r
            done       = terminated or truncated
        ppo_rewards.append(total_r)

    ppo_eval_env.close()
    ppo_mean = stats.mean(ppo_rewards)
    ppo_std  = stats.stdev(ppo_rewards)
    print(f"PPO ({SB3_EVAL_EPS} eps) — mean reward: {ppo_mean:.3f} ± {ppo_std:.3f}")

stable_baselines3 not installed — skipping SB3 cell.
Install with: pip install stable-baselines3


## Run Experiments — DQN & DDQN

In [ ]:
# On CPU use a lighter budget; GPU/MPS uses the full config
cfg = DQNConfig() if str(device) == "cpu" else DQNConfig(num_episodes=200_000)

SEEDS = [0, 1, 2, 3, 4]

dqn_results,  dqn_all_rewards  = [], []
ddqn_results, ddqn_all_rewards = [], []

for run_seed in SEEDS:
    print(f"\n{'='*50}\nSeed {run_seed}\n{'='*50}")

    for AgentClass, results_list, rewards_list, label in [
        (DQNAgent,  dqn_results,  dqn_all_rewards,  "DQN"),
        (DDQNAgent, ddqn_results, ddqn_all_rewards, "DDQN"),
    ]:
        env   = make_env(run_seed)
        agent = AgentClass(env, cfg, run_seed)

        print(f"  Training {label}…")
        rewards, lengths = agent.train(plot_fn=plot_training)

        eval_env = make_env(run_seed + 1000)
        mean_r, std_r, mean_len = evaluate_policy(
            eval_env, agent.policy_net, n_episodes=50, seed_offset=run_seed + 2000
        )
        eval_env.close()
        env.close()

        results_list.append({"seed": run_seed, "mean": mean_r, "std": std_r, "length": mean_len})
        rewards_list.append(rewards)
        print(f"  {label} seed={run_seed} | mean={mean_r:.3f} ± {std_r:.3f} | len={mean_len:.1f}")

print("\nAll experiments done.")

## Results

In [ ]:
print("### DQN results (50 eval episodes per seed)")
print("| seed | mean reward | std  | mean length |")
print("| ---- | ----------- | ---- | ----------- |")
for r in dqn_results:
    print(f"| {r['seed']}    | {r['mean']:+.3f}       | {r['std']:.3f} | {r['length']:.1f}       |")

dqn_grand_mean = stats.mean(r['mean'] for r in dqn_results)
dqn_grand_std  = stats.mean(r['std']  for r in dqn_results)
print(f"| **avg** | **{dqn_grand_mean:+.3f}** | **{dqn_grand_std:.3f}** | — |")

print()
print("### DDQN results")
print("| seed | mean reward | std  | mean length |")
print("| ---- | ----------- | ---- | ----------- |")
for r in ddqn_results:
    print(f"| {r['seed']}    | {r['mean']:+.3f}       | {r['std']:.3f} | {r['length']:.1f}       |")

ddqn_grand_mean = stats.mean(r['mean'] for r in ddqn_results)
ddqn_grand_std  = stats.mean(r['std']  for r in ddqn_results)
print(f"| **avg** | **{ddqn_grand_mean:+.3f}** | **{ddqn_grand_std:.3f}** | — |")

if SB3_AVAILABLE:
    print()
    print(f"### SB3 PPO: {ppo_mean:+.3f} ± {ppo_std:.3f}  ({SB3_EVAL_EPS} episodes)")

In [ ]:
plot_multi_seed(dqn_all_rewards,  [f"DQN  seed={s}" for s in SEEDS], title="DQN  — training curves")
plot_multi_seed(ddqn_all_rewards, [f"DDQN seed={s}" for s in SEEDS], title="DDQN — training curves")

# Side-by-side mean comparison
fig, ax = plt.subplots(figsize=(10, 4))
for rewards, label, color in [
    (dqn_all_rewards,  "DQN",  "steelblue"),
    (ddqn_all_rewards, "DDQN", "darkorange"),
]:
    max_ep = max(len(r) for r in rewards)
    padded = np.full((len(rewards), max_ep), np.nan)
    for i, r in enumerate(rewards):
        padded[i, :len(r)] = r
    mean = np.nanmean(padded, axis=0)
    std  = np.nanstd(padded, axis=0)
    xs   = np.arange(max_ep)
    ax.plot(xs, mean, label=label, color=color)
    ax.fill_between(xs, mean - std, mean + std, alpha=0.15, color=color)

ax.set_title("DQN vs DDQN — mean ± std across seeds")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.legend()
plt.tight_layout()
plt.show()

## Failure Mode Analysis

We run a greedy evaluation with collision tracking to quantify crash rates.


In [ ]:
def evaluate_with_crash_rate(env: gym.Env, policy_net: nn.Module,
                              n_episodes: int, seed_offset: int) -> dict:
    """Extended eval that also reports crash and timeout rates."""
    policy_net.eval()
    crashes, timeouts, rewards = 0, 0, []

    with torch.no_grad():
        for i in range(n_episodes):
            state, _ = env.reset(seed=seed_offset + i)
            state     = to_state_tensor(state)
            total_r   = 0.0

            for t in count():
                action = policy_net(state).max(1).indices.view(1, 1)
                obs, reward, terminated, truncated, info = env.step(action.item())
                total_r += reward

                if terminated:
                    crashed = info.get("crashed", False)
                    if crashed:
                        crashes += 1
                    break
                if truncated:
                    timeouts += 1
                    break
                state = to_state_tensor(obs)

            rewards.append(total_r)

    policy_net.train()
    return {
        "mean_reward": stats.mean(rewards),
        "crash_rate":   crashes  / n_episodes,
        "timeout_rate": timeouts / n_episodes,
    }

# Run on the best DQN checkpoint from each seed
print("Failure mode analysis — DQN (first seed as example)")
crash_env   = make_env(SEEDS[0] + 1000)
best_agent  = DQNAgent(crash_env, cfg, SEEDS[0])   # placeholder — load best ckpt in practice
crash_stats = evaluate_with_crash_rate(crash_env, best_agent.policy_net, 50, SEEDS[0] + 5000)
crash_env.close()

print(f"  mean reward : {crash_stats['mean_reward']:.3f}")
print(f"  crash rate  : {crash_stats['crash_rate']*100:.1f}%")
print(f"  timeout rate: {crash_stats['timeout_rate']*100:.1f}%")
print()
print("Known failure modes observed during training:")
print("  1. Merging into dense traffic — agent attempts lane change without sufficient gap.")
print("  2. High-speed tail-gating    — reward bonus for speed overrides braking behaviour.")
print("  3. Edge-lane deadlock        — agent hugs the rightmost lane and cannot overtake.")